In [1]:
import ast
import json
import re
import requests

import pandas as pd
import numpy as np

from bs4 import BeautifulSoup

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 500)

url = 'https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC' # Mapa de la Comunidad de Madrid y alrededores
df = pd.read_csv('../data/pisos_madrid.csv', sep ='|') # NO TENEMOS QUE HACERLO DESPUES DEL SCRAPING ? 

# Extracción de datos

In [2]:
def extraer_informacion(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    estate_component = soup.find("estate-show-v2")
    if not estate_component:
        return []
    
    estate_json_raw = estate_component.get(":estate")
    estate_json_raw = estate_json_raw.replace("&quot;", '"')
    estate_data = json.loads(estate_json_raw)

    data = {
        'dormitorios': estate_data.get('rooms'),
        'superficie_m2': estate_data.get('numeric_surface'),
        'baños': estate_data.get('bathrooms'),
        'url': estate_data.get('detail_url'),
        'features': estate_data.get('features'),
        'descripcion': estate_data.get('description'),
        'precio': estate_data.get('costs'),
        'latitud': estate_data.get('latitude'),
        'longitud': estate_data.get('longitude'),
        'media': estate_data.get('media'),
        'points_of_interest': estate_data.get('points_of_interest'),
        'energy_data': estate_data.get('energy_data'),
    }
    
    return data

In [3]:
def obtener_urls(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    print(f'Buscando pisos en la página {url} ...')
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")
    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')
    estates_data = json.loads(estates_raw)

    data = []

    for estate in estates_data:
        url_piso = estate.get("detail_url")

        if url_piso in df['url'].to_list():
            print('Ya lo tengo:', url_piso)
            return data
        data.append(extraer_informacion(url_piso))

    return data

In [4]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
ultima = soup.find('a', string='>>')
max_pages = int(re.findall(r'pag-(\d+)', str(ultima))[0])

url_splited = url.split('pag-1')
data = []
for i in range(1, max_pages+1):
    subdata = obtener_urls(f'pag-{i}'.join(url_splited))
    data.extend(subdata)

    if len(subdata) < 15:
        break

df = pd.concat([pd.DataFrame(data), df])
df.to_csv('../data/pisos_madrid.csv', sep='|', index=False)

Buscando pisos en la página https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC ...
Ya lo tengo: https://www.tecnocasa.es/venta/piso/toledo/cobisa/646306.html


# Pretratamiento de datos

In [5]:
# A partir de este punto sería interesante integrar un pipeline, para que si entra un nuevo piso todo se ejecute en el orden adecuado.

In [24]:
def _safe_eval(x):
    if isinstance(x, (dict, list)):
        return x
    if pd.isna(x):
        return {}
    s = str(x).strip()
    if s in ("", "{}", "[]", "None", "nan"):
        return {}
    try:
        return ast.literal_eval(s)
    except Exception:
        return {}

def parse_nested_tecnocasa(X: pd.DataFrame) -> pd.DataFrame:
    df = X.copy()

    # 1) parseo seguro de columnas tipo dict/list que vienen como string
    for c in ["features", "precio", "media", "points_of_interest", "energy_data"]:
        if c in df.columns:
            df[c] = df[c].apply(_safe_eval)

    # 2) features
    if "features" in df.columns:
        f = df["features"]
        df["planta"] = f.apply(lambda d: d.get("floor"))
        df["aire_acondicionado"] = f.apply(lambda d: d.get("air_conditioning"))
        df["ascensor"] = f.apply(lambda d: d.get("elevator"))
        df["calefaccion"] = f.apply(lambda d: d.get("heating"))
        df["categoria"] = f.apply(lambda d: d.get("category"))
        df["año_construccion"] = f.apply(lambda d: d.get("build_year"))

    # 3) precio
    if "precio" in df.columns:
        p = df["precio"]
        df["precio_euros"] = p.apply(lambda d: d.get("price") if isinstance(d, dict) else np.nan)

        df["precio_euros"] = pd.to_numeric(p.apply(lambda d: d.get("price") if isinstance(d, dict) else np.nan)
                                           .astype(str).str.replace("€", "", regex=False).str.replace(" ", "", regex=False)
                                           .str.replace(".", "", regex=False).str.replace(",", ".", regex=False),errors="coerce" )
        df["hipoteca"] = p.apply(lambda d: d.get("mortgage_payment_value") if isinstance(d, dict) else np.nan).str.replace(".", "", regex=False).astype(int, errors="ignore")

    # 4) media
    if "media" in df.columns:
        m = df["media"]
        df["planos"] = m.apply(lambda d: d.get("floor_plans") is not None)
        df["realistico"] = m.apply(lambda d: d.get("has_realistico"))
        df["fotografias"] = m.apply(lambda d: len(d.get("images", [])) if isinstance(d.get("images", []), list) else 0)

    # 5) points_of_interest
    if "points_of_interest" in df.columns:
        poi = df["points_of_interest"]
        df["transporte_publico"] = poi.apply(lambda d: d.get("public_transport"))
        df["escuelas"] = poi.apply(lambda d: d.get("school"))
        df["farmacias"] = poi.apply(lambda d: d.get("pharmacy"))
        df["hospitales"] = poi.apply(lambda d: d.get("hospital"))
        df["supermercados"] = poi.apply(lambda d: d.get("market"))
        df["tiendas"] = poi.apply(lambda d: d.get("shop"))
        df["bares"] = poi.apply(lambda d: d.get("bar"))
        df["restaurantes"] = poi.apply(lambda d: d.get("restaurant"))

    # 6) energy_data
    if "energy_data" in df.columns:
        e = df["energy_data"]
        df["clase_energetica"] = e.apply(lambda d: d.get("class_emissions"))
        df["eficiencia_energetica"] = e.apply(lambda d: d.get("efficiency"))
        df["emisiones_energeticas"] = e.apply(lambda d: d.get("emissions"))

    return df


parse_step = FunctionTransformer(parse_nested_tecnocasa)

In [9]:
df = parse_nested_tecnocasa(df)

In [25]:
POI_MAP = {
    "transporte_publico": "tp",
    "escuelas": "esc",
    "farmacias": "fca",
    "hospitales": "hosp",
    "supermercados": "super",
    "tiendas": "tda",
    "bares": "bar",
    "restaurantes": "resto",
}

def _to_m(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.nan
    t = str(s).lower().replace(",", ".").replace(" ", "")
    if "km" in t:
        return float(t.replace("km", "")) * 1000
    if "m" in t:
        return float(t.replace("m", ""))
    return np.nan

def poi_features_tecnocasa(X: pd.DataFrame) -> pd.DataFrame:
    df = X.copy()

    for col, pref in POI_MAP.items():
        if col not in df.columns:
            continue

        df[f"{pref}_cnt"] = df[col].apply(lambda lst: len(lst) if lst else 0)

        df[f"{pref}_min_dist_m"] = df[col].apply(
            lambda lst: np.nan if not lst else min(
                _to_m(d.get("distance"))
                for d in lst
                if isinstance(d, dict) and d.get("distance")
            )
        )

    return df

poi_step = FunctionTransformer(poi_features_tecnocasa)

In [9]:
df = poi_features_tecnocasa(df)

In [26]:
def final_clean_tecnocasa(X: pd.DataFrame) -> pd.DataFrame:
    df = X.copy()

    # --- Dormitorios / baños: pasar a numérico ---
    if "dormitorios" in df.columns:
        df["dormitorios"] = (df["dormitorios"]
                             .astype(str)
                             .str.replace(r"\s*dorm\.", "", regex=True)
                             .str.strip())
        df["dormitorios"] = pd.to_numeric(df["dormitorios"], errors="coerce")

    if "baños" in df.columns:
        df["baños"] = (df["baños"]
                       .astype(str)
                       .str.replace(r"\s*baño[s]*", "", regex=True)
                       .str.strip())
        df["baños"] = pd.to_numeric(df["baños"], errors="coerce")

    # --- Planta: sacar paréntesis (ej: "3 (Exterior)" -> "3") ---
    if "planta" in df.columns:
        df["planta"] = (df["planta"]
                        .astype(str)
                        .str.replace(r"\s*\(.*\)", "", regex=True)
                        .str.strip())

    # --- Aire acondicionado: limpieza simple de texto ---
    if "aire_acondicionado" in df.columns:
        df["aire_acondicionado"] = (df["aire_acondicionado"]
                                    .astype(str)
                                    .str.replace(r"\s+.+", "", regex=True)
                                    .str.strip())

    # --- Calefacción: flags y limpieza ---
    if "calefaccion" in df.columns:
        cal = df["calefaccion"].astype(str)
        df["calefaccion_gas"] = cal.str.contains("gas", case=False, na=False)
        df["calefaccion_electrica"] = cal.str.contains("eléctrica|electrica", case=False, na=False)
        df["calefaccion"] = cal.str.replace(r"\s+.+", "", regex=True).str.strip()

    # --- Energía: limpiar comas / espacios raros ---
    if "eficiencia_energetica" in df.columns:
        s = df["eficiencia_energetica"].astype(str).str.lower().str.replace(",", ".", regex=False)
        df["eficiencia_energetica"] = pd.to_numeric(s.str.extract(r"(\d+(\.\d+)?)")[0], errors="coerce")

    if "emisiones_energeticas" in df.columns:
        df["emisiones_energeticas"] = (df["emisiones_energeticas"]
                                       .astype(str)
                                       .str.replace(",", ".", regex=False)
                                       .replace({"nan": np.nan, "None": np.nan, "": np.nan})).astype(float,errors="ignore")

    # --- Booleanos: asegurar 0/1 en ascensor, planos, etc. (si existen) ---
    for b in ["ascensor", "planos", "realistico", "calefaccion_gas", "calefaccion_electrica"]:
        if b in df.columns:
            df[b] = (
                df[b]
                .replace({"True": True, "False": False, "true": True, "false": False, "Sí": True, "Si": True, "No": False, "1": True, "0": False})
                .fillna(False)
                .astype(bool)
                .astype(int))

    
    df = df.replace({"None": np.nan, "nan": np.nan, "": np.nan, "NA": np.nan, "N/A": np.nan})

    return df

final_clean_step = FunctionTransformer(final_clean_tecnocasa)

In [ ]:
df = final_clean_tecnocasa(df)

,dormitorios,superficie_m2,baños,url,features,descripcion,precio,latitud,longitud,media,points_of_interest,energy_data,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,transporte_publico,escuelas,farmacias,hospitales,supermercados,tiendas,bares,restaurantes,clase_energetica,eficiencia_energetica,emisiones_energeticas,tp_cnt,tp_min_dist_m,esc_cnt,esc_min_dist_m,fca_cnt,fca_min_dist_m,hosp_cnt,hosp_min_dist_m,super_cnt,super_min_dist_m,tda_cnt,tda_min_dist_m,bar_cnt,bar_min_dist_m,resto_cnt,resto_min_dist_m,calefaccion_gas,calefaccion_electrica
0,1.0,50.00,1.0,https://www.tecnocasa.es/venta/piso/madrid/las...,"{'id': 648733, 'floor': '', 'floors': None, 'b...",<p>Inmobiliaria en Las Rozas De Madrid - Tecno...,"{'price': '327.500 €', 'box_price': None, 'mor...",40.492402,-3.878863,"{'floor_plans': None, 'has_realistico': False,...",{},"{'class': '', 'class_emissions': None, 'certif...",NaN,NaN,0,Independiente,Popular,1991,327.500 €,1.055,0,0,36,None,None,None,None,None,None,None,None,None,NaN,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,1,0
1,3.0,91.00,2.0,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 649824, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de MADRID - zona Aveni...,"{'price': '565.000 €', 'box_price': None, 'mor...",40.480702,-3.713013,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'Avenida de la ...,"{'class': 'd', 'class_emissions': 'd', 'certif...",Baja,NaN,1,centralizada,Popular,1974,565.000 €,1.820,0,0,40,"[{'name': 'Avenida de la Ilustración', 'class'...","[{'name': 'Peque's School', 'class': 'school',...",[{'name': 'Farmacia - Calle Fermín Caballero 7...,"[{'name': 'Clínica Ruber Internacional', 'clas...","[{'name': 'Supersol', 'class': 'grocery', 'sub...","[{'name': 'La Tahona', 'class': 'shop', 'subcl...","[{'name': 'Hemeya', 'class': 'bar', 'subclass'...","[{'name': 'La Mesa Puesta', 'class': 'restaura...",d,172.0,36,5,110.0,5,120.0,5,130.0,5,950.0,5,140.0,5,70.0,5,70.0,5,50.0,1,0
2,7.0,190.00,3.0,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 640542, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de MADRID - zona Aveni...,"{'price': '990.000 €', 'box_price': None, 'mor...",40.480602,-3.717203,"{'floor_plans': [{'id': 9600945, 'width': 683,...",{'public_transport': [{'name': 'Avenida de la ...,"{'class': 'd', 'class_emissions': 'd', 'certif...",Baja,NaN,0,Independiente,Señorial,1968,990.000 €,3.189,1,0,30,"[{'name': 'Avenida de la Ilustración', 'class'...","[{'name': 'Colegio Público Bravo Murillo', 'cl...","[{'name': 'Farmacia - Calle Isla de Arosa 43',...","[{'name': 'Clínica Ruber Internacional', 'clas...","[{'name': 'Unide Market', 'class': 'grocery', ...","[{'name': 'Tiendas', 'class': 'shop', 'subclas...","[{'name': 'Hemeya', 'class': 'bar', 'subclass'...","[{'name': 'Bar Restaurante La Ponderosa', 'cla...",d,124.2,25.8,5,20.0,5,110.0,5,30.0,5,940.0,5,40.0,5,130.0,5,380.0,5,80.0,1,0
3,3.0,68.00,1.0,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 598798, 'floor': '', 'floors': None, 'b...","<p>Reforma a estrenar, estancias exteriores e ...","{'price': '290.000 €', 'box_price': None, 'mor...",40.432702,-3.640053,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Ascao', 'class...","{'class': '', 'class_emissions': None, 'certif...",NaN,NaN,0,Independiente,Media,1960,290.000 €,934,0,0,24,"[{'name': 'Ascao', 'class': 'railway', 'subcla...","[{'name': 'Colegio Joyfe', 'class': 'school', ...","[{'name': 'Farmacia - Calle Vital Aza 59', 'cl...","[{'name': 'Clínica Fuensanta', 'class': 'hospi...","[{'name': 'Ahorramás', 'class': 'grocery', 'su...","[{'name': 'Levain', 'class': 'shop', 'subclass...","[{'name': 'Los Tres Diamantes', 'class': 'bar'...","[{'name': 'Piscis', 'class': 'restaurant', 'su...",None,NaN,NaN,4,220.0,5,60.0,5,90.0,5,720.0,5,140.0,5,290.0,5,310.0,5,330.0,1,0
0,3.0,90.0,NaN,http

In [27]:
df_raw = pd.read_csv('../data/pisos_madrid.csv', sep ='|')

DROP_COLS = ['url', 'features', 'descripcion', 'precio', 'media', 'points_of_interest',
             'energy_data', 'transporte_publico', 'escuelas', 'farmacias', 'hospitales',
             'supermercados', 'tiendas', 'bares', 'restaurantes']
drop_step = FunctionTransformer(
    lambda df: df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
)

In [28]:
pipe_features = Pipeline([
    ("parse_nested", parse_step),        # crea transporte_publico, escuelas, precio_euros, etc.
    ("poi_features", poi_step),          # crea tp_cnt, tp_min_dist_m, esc_cnt, ...
    ("final_clean", final_clean_step),   # regex, flags, booleans, normalización de NaN
    ("drop",drop_step)
])

df_feat = pipe_features.fit_transform(df_raw)

In [31]:
df_feat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1206 entries, 0 to 1205
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dormitorios            1096 non-null   float64
 1   superficie_m2          1206 non-null   float64
 2   baños                  1110 non-null   float64
 3   latitud                1206 non-null   float64
 4   longitud               1206 non-null   float64
 5   planta                 760 non-null    object 
 6   aire_acondicionado     450 non-null    object 
 7   ascensor               1206 non-null   int64  
 8   calefaccion            669 non-null    object 
 9   categoria              1206 non-null   object 
 10  año_construccion       1206 non-null   int64  
 11  precio_euros           1206 non-null   int64  
 12  hipoteca               1206 non-null   int64  
 13  planos                 1206 non-null   int64  
 14  realistico             1206 non-null   int64  
 15  foto

In [24]:
# Separar en X_train, X_test, y_train , y_test
# Otro pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('nombre', funcion, ['columnas']),
    ],
    remainder="drop"
)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_sel),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_sel),
    ],
    remainder="drop"
)